# LTX Audio-Only Single-Stream Walkthrough

This notebook follows the real `ltx_core` audio-only path end-to-end: environment setup, imports, tiny model construction, synthetic `Modality` setup, `prepare()`, a single transformer block, and the full `LTXModel.forward()` call.

## 1. Environment Setup

Make sure the notebook can import directly from the local `ltx-core` source tree.

In [ ]:
from pathlib import Path
import sys

START_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in [START_DIR, *START_DIR.parents] if (path / "packages" / "ltx-core" / "src").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing packages/ltx-core/src")

NOTEBOOK_DIR = REPO_ROOT / "packages" / "ltx-core" / "notebooks"
LTX_CORE_SRC = REPO_ROOT / "packages" / "ltx-core" / "src"
if str(LTX_CORE_SRC) not in sys.path:
    sys.path.insert(0, str(LTX_CORE_SRC))

print("Started from:", START_DIR)
print("Notebook dir:", NOTEBOOK_DIR)
print("Repository root:", REPO_ROOT)
print("ltx-core src:", LTX_CORE_SRC)


## 2. Source Map

This walkthrough is intentionally source-mapped to the real audio-only path:

1. `packages/ltx-core/src/ltx_core/model/transformer/modality.py` defines the `Modality` container we construct below.
2. `packages/ltx-core/src/ltx_core/model/transformer/transformer_args.py` defines the prepared transformer argument structure returned by `audio_args_preprocessor.prepare()`.
3. `packages/ltx-core/src/ltx_core/model/transformer/transformer.py` contains the transformer block logic stepped through in the single-block cell.
4. `packages/ltx-core/src/ltx_core/model/transformer/model.py` wires the audio-only model together and owns the full `LTXModel.forward()` path.

## 3. Imports

Pull in the real classes used by the audio-only transformer path.

In [ ]:
import torch

from ltx_core.guidance.perturbations import BatchedPerturbationConfig
from ltx_core.model.transformer.model import LTXModel, LTXModelType
from ltx_core.model.transformer.modality import Modality


## 4. Build a Tiny AudioOnly Model

Construct a very small `AudioOnly` model so the rest of the walkthrough stays easy to inspect.

In [ ]:
torch.manual_seed(0)

model = LTXModel(
    model_type=LTXModelType.AudioOnly,
    num_layers=2,
    audio_num_attention_heads=2,
    audio_attention_head_dim=8,
    audio_in_channels=12,
    audio_out_channels=12,
    audio_cross_attention_dim=16,
    cross_attention_adaln=False,
)
model.eval()
print(model.model_type)
print("audio_inner_dim =", model.audio_inner_dim)
print("num_blocks =", len(model.transformer_blocks))


## 5. Construct a Minimal Fake Audio Modality

Build a tiny batch of fake latent/audio context inputs that still match the real model API.

`positions` uses shape `(batch, 1, tokens, 2)` so each audio token carries the paired positional coordinates expected by the audio-only path.
`context_mask` and `attention_mask` are all-ones masks here because this walkthrough exercises the full text context and full token-to-token self-attention surface without padding.

In [ ]:
BATCH = 2
TOKENS = 6
IN_CHANNELS = 12
TEXT_TOKENS = 4
TEXT_DIM = 16

latent = torch.randn(BATCH, TOKENS, IN_CHANNELS)
sigma = torch.full((BATCH,), 0.5)
timesteps = torch.full((BATCH, TOKENS), 0.25)
positions = torch.arange(TOKENS, dtype=torch.float32).view(1, 1, TOKENS, 1).repeat(BATCH, 1, 1, 2)
positions[..., 1] = positions[..., 0] + 1
context = torch.randn(BATCH, TEXT_TOKENS, TEXT_DIM)
context_mask = torch.ones(BATCH, 1, TEXT_TOKENS)
attention_mask = torch.ones(BATCH, TOKENS, TOKENS)

audio_modality = Modality(
    latent=latent,
    sigma=sigma,
    timesteps=timesteps,
    positions=positions,
    context=context,
    enabled=True,
    context_mask=context_mask,
    attention_mask=attention_mask,
)
audio_modality


## 6. Run the Real `prepare()` Path

This cell uses the real `audio_args_preprocessor.prepare(...)` to convert `Modality` into the transformer args object.

In [ ]:
audio_args = model.audio_args_preprocessor.prepare(audio_modality, None)

print("x:", tuple(audio_args.x.shape))
print("context:", tuple(audio_args.context.shape))
print("timesteps:", tuple(audio_args.timesteps.shape))
print("embedded_timestep:", tuple(audio_args.embedded_timestep.shape))
print("self_attention_mask:", None if audio_args.self_attention_mask is None else tuple(audio_args.self_attention_mask.shape))


## 7. Run One Real Transformer Block

This cell runs one real `BasicAVTransformerBlock` so you can inspect the shape contract before the full model forward.

In [ ]:
block = model.transformer_blocks[0]
_, audio_after_block = block(video=None, audio=audio_args)
print("block input x:", tuple(audio_args.x.shape))
print("block output x:", tuple(audio_after_block.x.shape))


## 8. Run Full `LTXModel.forward()`

This cell runs the full real `LTXModel.forward()` audio-only path with an empty perturbation config.

In [ ]:
perturbations = BatchedPerturbationConfig.empty(BATCH)
vx, ax = model(video=None, audio=audio_modality, perturbations=perturbations)
print("video output is None for AudioOnly:", vx is None)
print("audio output shape:", tuple(ax.shape))


## 9. Reading Guide

Next source files to inspect:

1. `packages/ltx-core/src/ltx_core/model/transformer/modality.py`
2. `packages/ltx-core/src/ltx_core/model/transformer/transformer_args.py`
3. `packages/ltx-core/src/ltx_core/model/transformer/transformer.py`
4. `packages/ltx-core/src/ltx_core/model/transformer/model.py`
